TabNet - Interpretable Deep Tabular Model for PEAD Prediction

Attention-based architecture (Arik & Pfister, 2021) with built-in
feature selection. Separate classifier (3-class) and regressor
(peak return magnitude) - TabNet doesn't natively support a shared
multi-task head, so this is two independent models trained on the
same folds, evaluated the same quarterly expanding-window
walk-forward as the Multi-Task DNN.

**How this notebook is organized:**
0. Setup - mount Drive, imports, device/seed config
1. Data Loading - same join as the DNN notebook
2. Feature List - flatten everything into one input vector
3. Fold Construction - quarterly walk-forward train/val/test splits
4. Architecture & Training Functions - TabNet classifier + regressor
5. Hyperparameter Search - two-phase Optuna search, final tuned run
6. Visualizations - accuracy over time, confusion matrix, return spread
7. TabNet Attention Masks - feature importance, no SHAP needed
8. Save Results - pickle outputs for cross-pipeline comparison

## Setup

Mounts Google Drive so the notebook can read the modeling parquet
files and, at the end, write the results pickle back to Drive.

In [1]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Imports, Device, and Reproducibility

Same environment setup as the Multi-Task DNN notebook, swapping in
`pytorch_tabnet` for the model itself (`TabNetClassifier`,
`TabNetRegressor`) instead of raw `torch.nn`. `set_seed()` is called
at the start of every training run for reproducibility across folds.

In [4]:

import polars as pl
import numpy as np
import os, pickle, re, random, time

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score,
)

import torch
from pytorch_tabnet.tab_model import TabNetClassifier, TabNetRegressor
import matplotlib.pyplot as plt

!pip install optuna -q
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = 'white'
plt.rcParams['text.color']       = 'black'
plt.rcParams['axes.labelcolor']  = 'black'
plt.rcParams['xtick.color']      = 'black'
plt.rcParams['ytick.color']      = 'black'
plt.rcParams['axes.edgecolor']   = 'black'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 12.1 MB/s eta 0:00:00
Using device: cuda


## 1. Data Loading

Identical join logic to the DNN notebook (tech + fundamentals + FinBERT transcript sentiment + news sentiment + sector one-hot), so both models see exactly the same information.

In [5]:
"""## 1. Data Loading

Joins tech indicators + fundamentals + FinBERT transcript sentiment +
news sentiment + sector one-hot into a single modeling table, keyed on
(symbol, earnings_date).
"""

df_tech = pl.read_parquet("/content/tech_modeling_table.parquet")
print(f"Tech table shape: {df_tech.shape}")

df_fund = pl.read_parquet("/content/modeling_fundamentals.parquet")
df_fund = df_fund.select([
    "symbol",
    pl.col("reportedDate").alias("earnings_date"),
    "eps_growth_qoq", "revenue_growth_qoq",
    "gross_margin",     "gross_margin_qoq",
    "debt_to_equity",   "debt_to_equity_qoq",
    "fcf_margin",       "fcf_margin_qoq",
    "roe",              "roe_qoq",
    "surprisePercentage",
])

df_finbert_raw = pl.read_parquet("/content/finbert_tx_agg_weighted.parquet")

df_finbert = df_finbert_raw.select([
    "symbol",
    pl.col("reportedDate").alias("earnings_date"),
    "pos_prob", "neg_prob",
])

df_nz = pl.read_parquet("/content/nz_sentiment.parquet")
df_nz = df_nz.select([
    "symbol",
    pl.col("reportedDate").alias("earnings_date"),
    "overall_sentiment_score_pre",  "ticker_sentiment_score_pre",
    "overall_sentiment_score_post", "ticker_sentiment_score_post",
])

df_model = df_tech.join(df_fund,     on=["symbol", "earnings_date"], how="left")
df_model = df_model.join(df_finbert, on=["symbol", "earnings_date"], how="left")
df_model = df_model.join(df_nz,      on=["symbol", "earnings_date"], how="left")

df_sector = df_finbert_raw.select(["symbol", "sector"]).unique()
df_sector = df_sector.with_columns(pl.col("sector").fill_null("Unknown"))
sectors = sorted(df_sector["sector"].unique().to_list())
df_sector = df_sector.with_columns([
    (pl.col("sector") == s).cast(pl.Int8).alias(f"sector_{s.replace(' ', '_')}")
    for s in sectors
]).drop("sector")
df_model = df_model.join(df_sector, on="symbol", how="left")

df_model = df_model.drop([
    col for col in df_model.columns if col.startswith("car")
])

print(f"Combined table shape: {df_model.shape}")
print(f"Target class distribution:\n{df_model['target_class'].value_counts().sort('target_class')}")

Tech table shape: (24048, 239)
Combined table shape: (24048, 266)
Target class distribution:
shape: (3, 2)
┌──────────────┬───────┐
│ target_class ┆ count │
│ ---          ┆ ---   │
│ i64          ┆ u32   │
╞══════════════╪═══════╡
│ 0            ┆ 7955  │
│ 1            ┆ 6428  │
│ 2            ┆ 9665  │
└──────────────┴───────┘


## 2. Feature List

Same flat, single-vector feature layout as the DNN, no separate encoders per modality (that was tried in the fusion-DNN experiment and underperformed). The tech/VIX sequence columns are kept contiguous and ordered by timestep purely so they can be sliced out later if needed (e.g. for the attention-by-modality analysis below); TabNet itself treats all 257 columns as one flat input, same as the DNN.

In [ ]:
"""## 2. Feature List (Flat, Single Encoder)

All features concatenated into one flat vector - no per-modality
encoders (tested separately, underperformed).
"""

EXCLUDE_COLS = [
    "symbol", "earnings_date", "entry_price", "target_return",
    "target_class", "max_high", "min_high", "max_day", "min_day", "year", "quarter",
]
feature_cols = [c for c in df_model.columns if c not in EXCLUDE_COLS]

TECH_BASES = [
    "rsi", "macd", "macd_hist", "roc",
    "ema50_pct", "ema200_pct", "ema50_200_pct", "adx",
    "atr", "bb_width", "bb_pct_b", "sigma",
    "obv_zscore", "vwap_pct",
    "open_pct", "high_pct", "low_pct", "volume_rel",
]
VIX_BASES = ["vix_close"]
SEQ_BASES = TECH_BASES + VIX_BASES  # 19 indicators × 12 timesteps

def group_sequence_columns(all_cols, bases):
    groups = {}
    remaining = list(all_cols)
    for base in sorted(bases, key=len, reverse=True):
        matched = [c for c in remaining if c == base or c.startswith(base + "_")]
        def step_key(col, base=base):
            m = re.search(r"(-?\d+)", col[len(base):])
            return int(m.group(1)) if m else 0
        groups[base] = sorted(matched, key=step_key)
        remaining = [c for c in remaining if c not in matched]
    return {base: groups[base] for base in bases}

seq_groups = group_sequence_columns(feature_cols, SEQ_BASES)
timestep_counts = {len(c) for c in seq_groups.values()}
assert len(timestep_counts) == 1, f"Inconsistent timestep counts: {timestep_counts}"
n_timesteps = timestep_counts.pop()
seq_cols_ordered = [c for base in SEQ_BASES for c in seq_groups[base]]

other_cols = [c for c in feature_cols if c not in seq_cols_ordered]

flat_feature_cols = seq_cols_ordered + other_cols

assert len(flat_feature_cols) == len(feature_cols), (
    f"Feature count mismatch: flat={len(flat_feature_cols)}, original={len(feature_cols)}"
)
print(f"Sequence block: {len(seq_cols_ordered)} cols ({len(SEQ_BASES)} indicators × {n_timesteps} steps)")
print(f"Other (fundamentals/sentiment/sector): {len(other_cols)} cols")
print(f"Total flat features: {len(flat_feature_cols)}")

Sequence block: 228 cols (19 indicators × 12 steps)
Other (fundamentals/sentiment/sector): 29 cols
Total flat features: 257


## 3. Fold Construction — Quarterly Expanding Window

Same expanding-window walk-forward as the Multi-Task DNN: for each
test quarter, train = everything before the validation quarter,
val = the single quarter immediately before test (needed here because
TabNet requires an eval set for its own early stopping), test = the
held-out quarter. Imputer and scaler are fit on the **training split
only** and applied unchanged to val/test — the leakage guard — and
the assertions in the code fail loudly if any date boundary overlaps.

In [8]:
"""## 3. Fold Construction — Quarterly Expanding Window

Same expanding window as Multi-Task DNN. TabNet requires a
validation set for early stopping, so the quarter immediately
before the test quarter is held out as validation. Training
uses everything before the validation quarter.

    Train: 2014 → val_quarter-1
    Val:   val_quarter (one quarter)
    Test:  test_quarter (one quarter)
"""

def prep_block(train_df, val_df, test_df, cols):
    """Fit imputer + scaler on train ONLY. Val and test are transform-only."""
    Xtr = train_df.select(cols).to_numpy()
    Xva = val_df.select(cols).to_numpy()
    Xte = test_df.select(cols).to_numpy()

    Xtr = np.where(np.isinf(Xtr), np.nan, Xtr)
    Xva = np.where(np.isinf(Xva), np.nan, Xva)
    Xte = np.where(np.isinf(Xte), np.nan, Xte)

    imputer = SimpleImputer(strategy="median").fit(Xtr)
    Xtr = imputer.transform(Xtr)
    Xva = imputer.transform(Xva)
    Xte = imputer.transform(Xte)

    scaler = StandardScaler().fit(Xtr)
    return scaler.transform(Xtr), scaler.transform(Xva), scaler.transform(Xte)


INITIAL_TRAIN_YEARS = 6

df_model = df_model.with_columns([
    pl.col("earnings_date").dt.year().alias("year"),
    pl.col("earnings_date").dt.quarter().alias("quarter"),
])

df_model = df_model.filter(pl.col("year") <= 2025)

min_date = df_model["earnings_date"].min()
max_date = df_model["earnings_date"].max()
print(f"Dataset spans: {min_date} to {max_date}")
print(f"Total rows: {len(df_model)}\n")

unique_quarters = df_model.select(["year", "quarter"]).unique().sort(["year", "quarter"])
quarters_list = unique_quarters.to_dicts()

print("=" * 100)
print("EXPANDING WINDOW FOLD CONSTRUCTION (val = quarter before test)")
print("=" * 100)

earliest_year = quarters_list[0]["year"]
initial_train_end_year = earliest_year + INITIAL_TRAIN_YEARS - 1  # 2019
first_test_idx = next(i for i, q in enumerate(quarters_list) if q["year"] > initial_train_end_year)

print(f"Initial training period: {earliest_year}–{initial_train_end_year}")
print(f"Testing begins at: {quarters_list[first_test_idx]}")
print("=" * 100 + "\n")

folds_data = []
fold_num = 1

for test_quarter_idx in range(first_test_idx, len(quarters_list)):
    test_q = quarters_list[test_quarter_idx]
    val_q  = quarters_list[test_quarter_idx - 1]
    test_label = f"{test_q['year']}_Q{test_q['quarter']}"

    test_data = df_model.filter(
        (pl.col("year") == test_q["year"]) & (pl.col("quarter") == test_q["quarter"])
    )

    val_data = df_model.filter(
        (pl.col("year") == val_q["year"]) & (pl.col("quarter") == val_q["quarter"])
    )

    train_data = df_model.filter(
        (pl.col("year") < val_q["year"]) |
        ((pl.col("year") == val_q["year"]) & (pl.col("quarter") < val_q["quarter"]))
    )

    if len(train_data) == 0 or len(val_data) == 0 or len(test_data) == 0:
        continue

    X_tr, X_va, X_te = prep_block(train_data, val_data, test_data, flat_feature_cols)

    folds_data.append({
        "fold_num":     fold_num,
        "test_quarter": test_label,
        "X_train":      X_tr,
        "X_val":        X_va,
        "X_test":       X_te,
        "y_train_cls":  train_data["target_class"].to_numpy(),
        "y_val_cls":    val_data["target_class"].to_numpy(),
        "y_test_cls":   test_data["target_class"].to_numpy(),
        "y_train_ret":  train_data["target_return"].to_numpy(),
        "y_val_ret":    val_data["target_return"].to_numpy(),
        "y_test_ret":   test_data["target_return"].to_numpy(),
    })

    print(f"Fold {fold_num:2d} [{test_label}]:  "
          f"train={X_tr.shape[0]:6,}  "
          f"val={X_va.shape[0]:4,} ({val_q['year']}_Q{val_q['quarter']})  "
          f"test={X_te.shape[0]:4,}")

    fold_num += 1

n_features = len(flat_feature_cols)
print("=" * 100)
print(f"Total folds: {len(folds_data)}")
print(f"n_features: {n_features}")
print("=" * 100)

Dataset spans: 2014-01-07 to 2025-12-19
Total rows: 23106

EXPANDING WINDOW FOLD CONSTRUCTION (val = quarter before test)
Initial training period: 2014–2019
Testing begins at: {'year': 2020, 'quarter': 1}

Fold  1 [2020_Q1]:  train=10,744  val= 486 (2019_Q4)  test= 485
Fold  2 [2020_Q2]:  train=11,230  val= 485 (2020_Q1)  test= 481
Fold  3 [2020_Q3]:  train=11,715  val= 481 (2020_Q2)  test= 487
Fold  4 [2020_Q4]:  train=12,196  val= 487 (2020_Q3)  test= 486
Fold  5 [2021_Q1]:  train=12,683  val= 486 (2020_Q4)  test= 492
Fold  6 [2021_Q2]:  train=13,169  val= 492 (2021_Q1)  test= 493
Fold  7 [2021_Q3]:  train=13,661  val= 493 (2021_Q2)  test= 493
Fold  8 [2021_Q4]:  train=14,154  val= 493 (2021_Q3)  test= 492
Fold  9 [2022_Q1]:  train=14,647  val= 492 (2021_Q4)  test= 494
Fold 10 [2022_Q2]:  train=15,139  val= 494 (2022_Q1)  test= 495
Fold 11 [2022_Q3]:  train=15,633  val= 495 (2022_Q2)  test= 493
Fold 12 [2022_Q4]:  train=16,128  val= 493 (2022_Q3)  test= 493
Fold 13 [2023_Q1]:  train=

## 4. TabNet — Architecture & Training Functions

TabNet (Arik & Pfister, 2021) does attention-based feature selection
across sequential decision steps, natively producing per-sample
feature importance without needing SHAP or permutation importance
(used later in Section 7). Unlike the Multi-Task DNN, TabNet doesn't
support a shared encoder with multiple heads, so classification and
regression are trained as two separate models here (`train_tabnet_clf`,
`train_tabnet_reg`) on the same fold data. Both use early stopping on
the validation quarter; regularization comes from `lambda_sparse`
(sparsity penalty on attention), entmax-masked attention, and gradient
clipping (`clip_value`).

In [ ]:

def train_tabnet_clf(f, n_d=32, n_a=32, n_steps=5, gamma=1.5,
                     lambda_sparse=1e-3, lr=2e-2, step_size=10,
                     scheduler_gamma=0.9, max_epochs=100, patience=15,
                     batch_size=512, clip_value=2.0, seed=42):
    """Train TabNet classifier with early stopping on validation quarter."""
    set_seed(seed)

    clf = TabNetClassifier(
        n_d=n_d, n_a=n_a, n_steps=n_steps, gamma=gamma,
        lambda_sparse=lambda_sparse,
        clip_value=clip_value,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=lr),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params=dict(step_size=step_size, gamma=scheduler_gamma),
        mask_type="entmax",
        verbose=0,
        seed=seed,
    )

    clf.fit(
        X_train=f["X_train"], y_train=f["y_train_cls"],
        eval_set=[(f["X_val"], f["y_val_cls"])],
        eval_name=["val"], eval_metric=["logloss"],
        max_epochs=max_epochs, patience=patience,
        batch_size=batch_size, virtual_batch_size=128,
        weights=1,  # automatic inverse-frequency class weighting
    )

    test_probs = clf.predict_proba(f["X_test"])
    test_preds = test_probs.argmax(axis=1)

    return {
        "clf_preds":  test_preds,
        "clf_probs":  test_probs,
        "best_epoch": clf.best_epoch,
        "model":      clf,
    }


def train_tabnet_reg(f, n_d=32, n_a=32, n_steps=5, gamma=1.5,
                     lambda_sparse=1e-3, lr=5e-3, step_size=10,
                     scheduler_gamma=0.9, max_epochs=150, patience=15,
                     batch_size=512, clip_value=2.0, seed=42):
    """Train TabNet regressor with early stopping on validation quarter."""
    set_seed(seed)

    reg = TabNetRegressor(
        n_d=n_d, n_a=n_a, n_steps=n_steps, gamma=gamma,
        lambda_sparse=lambda_sparse,
        clip_value=clip_value,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=lr),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params=dict(step_size=step_size, gamma=scheduler_gamma),
        mask_type="entmax",
        verbose=0,
        seed=seed,
    )

    y_mean, y_std = f["y_train_ret"].mean(), f["y_train_ret"].std()
    y_train_scaled = ((f["y_train_ret"] - y_mean) / y_std).reshape(-1, 1)
    y_val_scaled   = ((f["y_val_ret"]   - y_mean) / y_std).reshape(-1, 1)

    reg.fit(
        X_train=f["X_train"], y_train=y_train_scaled,
        eval_set=[(f["X_val"], y_val_scaled)],
        eval_name=["val"], eval_metric=["rmse"],
        max_epochs=max_epochs, patience=patience,
        batch_size=batch_size, virtual_batch_size=128,
    )

    test_preds_scaled = reg.predict(f["X_test"]).ravel()
    test_preds = (test_preds_scaled * y_std) + y_mean

    return {
        "reg_preds":  test_preds,
        "best_epoch": reg.best_epoch,
        "model":      reg,
    }

## 5. Hyperparameter Search — Expanding Walk-Forward with Optuna

Same two-phase structure as the Multi-Task DNN:
- **Phase 1** tunes on folds 1–4 (2020), evaluates on folds 5–12 (2021–2022).
- **Phase 2** retunes on folds 1–12 (2020–2022), evaluates on folds 13–24 (2023–2025).

The search (Optuna, TPE sampler) tunes the *classifier's* hyperparameters
(`n_d`, `n_a`, `n_steps`, `gamma`, `lambda_sparse`, `lr`, `max_epochs`).
The regressor reuses the same architecture params but trains at
1/4 the learning rate (`lr * 0.25`) — regression targets are noisier
and TabNet's regressor tends to diverge at the classifier's LR.

In [ ]:

N_TUNE_FOLDS = 4
RETUNE_AT = 12
N_TRIALS = 24

tune_folds_phase1 = folds_data[:N_TUNE_FOLDS]
eval_folds_phase1 = folds_data[N_TUNE_FOLDS:RETUNE_AT]
tune_folds_phase2 = folds_data[:RETUNE_AT]
eval_folds_phase2 = folds_data[RETUNE_AT:]
eval_folds = folds_data[N_TUNE_FOLDS:]

print(f"Phase 1 tune:  {[f['test_quarter'] for f in tune_folds_phase1]}")
print(f"Phase 1 eval:  {[f['test_quarter'] for f in eval_folds_phase1]}")
print(f"Phase 2 tune:  folds 1-{RETUNE_AT} "
      f"({tune_folds_phase2[0]['test_quarter']} to {tune_folds_phase2[-1]['test_quarter']})")
print(f"Phase 2 eval:  {[f['test_quarter'] for f in eval_folds_phase2]}")
print()


def run_optuna_search(tune_folds, n_features, n_trials=24):
    """Bayesian HP search (TPE) for TabNet classifier."""

    def objective(trial):
        cfg = {
            "n_d":            trial.suggest_categorical("n_d", [16, 32, 64]),
            "n_a":            trial.suggest_categorical("n_a", [16, 32, 64]),
            "n_steps":        trial.suggest_int("n_steps", 3, 7),
            "lambda_sparse":  trial.suggest_float("lambda_sparse", 1e-4, 1e-2, log=True),
            "lr":             trial.suggest_float("lr", 1e-3, 5e-2, log=True),
            "gamma":          trial.suggest_float("gamma", 1.0, 2.0),
            "max_epochs":     trial.suggest_categorical("max_epochs", [50, 100, 150]),
        }

        fold_aucs = []
        for tf in tune_folds:
            out = train_tabnet_clf(
                tf,
                n_d=cfg["n_d"], n_a=cfg["n_a"], n_steps=cfg["n_steps"],
                gamma=cfg["gamma"], lambda_sparse=cfg["lambda_sparse"],
                lr=cfg["lr"], max_epochs=cfg["max_epochs"],
                seed=42,
            )
            fold_aucs.append(roc_auc_score(
                tf["y_test_cls"], out["clf_probs"],
                multi_class="ovr", average="weighted"))

        return np.mean(fold_aucs)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42))

    enqueue_cfg = {
        "n_d": 32, "n_a": 32, "n_steps": 5, "gamma": 1.5,
        "lambda_sparse": 1e-3, "lr": 2e-2, "max_epochs": 100,
    }
    study.enqueue_trial(enqueue_cfg)

    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best = study.best_params
    best_cfg = {
        "n_d":           best["n_d"],
        "n_a":           best["n_a"],
        "n_steps":       best["n_steps"],
        "lambda_sparse": best["lambda_sparse"],
        "lr":            best["lr"],
        "gamma":         best["gamma"],
        "max_epochs":    best["max_epochs"],
    }

    print(f"  Best AUC: {study.best_value:.4f}")
    print(f"  Config:   {best_cfg}")

    return best_cfg, study


# ── Phase 1: tune on 2020, evaluate 2021–2022 ──────────────────────
print("=" * 80)
print(f"PHASE 1 — tuning on {len(tune_folds_phase1)} folds (2020)")
print("=" * 80)
best_cfg_p1, study_p1 = run_optuna_search(tune_folds_phase1, n_features, N_TRIALS)
print()

tabnet_clf = {"fold_acc": [], "preds": [], "probs": [], "true": [], "quarters": []}
tabnet_reg = {"fold_mae": [], "fold_rmse": [], "preds": [], "true": []}

for f in eval_folds_phase1:
    t0 = time.time()

    out_clf = train_tabnet_clf(
        f,
        n_d=best_cfg_p1["n_d"], n_a=best_cfg_p1["n_a"],
        n_steps=best_cfg_p1["n_steps"], gamma=best_cfg_p1["gamma"],
        lambda_sparse=best_cfg_p1["lambda_sparse"],
        lr=best_cfg_p1["lr"], max_epochs=best_cfg_p1["max_epochs"],
        seed=42 + f["fold_num"],
    )

    out_reg = train_tabnet_reg(
        f,
        n_d=best_cfg_p1["n_d"], n_a=best_cfg_p1["n_a"],
        n_steps=best_cfg_p1["n_steps"], gamma=best_cfg_p1["gamma"],
        lambda_sparse=best_cfg_p1["lambda_sparse"],
        lr=best_cfg_p1["lr"] * 0.25,  # regressor uses lower lr
        max_epochs=best_cfg_p1["max_epochs"],
        seed=42 + f["fold_num"],
    )

    acc = accuracy_score(f["y_test_cls"], out_clf["clf_preds"])
    tabnet_clf["fold_acc"].append(acc)
    tabnet_clf["preds"].extend(out_clf["clf_preds"])
    tabnet_clf["probs"].append(out_clf["clf_probs"])
    tabnet_clf["true"].extend(f["y_test_cls"])
    tabnet_clf["quarters"].append(f["test_quarter"])

    mae  = mean_absolute_error(f["y_test_ret"], out_reg["reg_preds"])
    rmse = np.sqrt(mean_squared_error(f["y_test_ret"], out_reg["reg_preds"]))
    tabnet_reg["fold_mae"].append(mae)
    tabnet_reg["fold_rmse"].append(rmse)
    tabnet_reg["preds"].extend(out_reg["reg_preds"])
    tabnet_reg["true"].extend(f["y_test_ret"])

    train_time = time.time() - t0
    pred_dist = np.bincount(out_clf["clf_preds"].astype(int), minlength=3)

    print(f"Fold {f['fold_num']:2d} [{f['test_quarter']}]: "
          f"DA={acc:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  "
          f"pred_dist={pred_dist}  ({train_time:.0f}s)")


# ── Phase 2: retune on 2020–2022, evaluate 2023–2025 ───────────────
print(f"\n{'=' * 80}")
print(f"PHASE 2 — retuning on {len(tune_folds_phase2)} folds (2020–2022)")
print(f"{'=' * 80}")
best_cfg_p2, study_p2 = run_optuna_search(tune_folds_phase2, n_features, N_TRIALS)
print()

for f in eval_folds_phase2:
    t0 = time.time()

    out_clf = train_tabnet_clf(
        f,
        n_d=best_cfg_p2["n_d"], n_a=best_cfg_p2["n_a"],
        n_steps=best_cfg_p2["n_steps"], gamma=best_cfg_p2["gamma"],
        lambda_sparse=best_cfg_p2["lambda_sparse"],
        lr=best_cfg_p2["lr"], max_epochs=best_cfg_p2["max_epochs"],
        seed=42 + f["fold_num"],
    )

    out_reg = train_tabnet_reg(
        f,
        n_d=best_cfg_p2["n_d"], n_a=best_cfg_p2["n_a"],
        n_steps=best_cfg_p2["n_steps"], gamma=best_cfg_p2["gamma"],
        lambda_sparse=best_cfg_p2["lambda_sparse"],
        lr=best_cfg_p2["lr"] * 0.25,
        max_epochs=best_cfg_p2["max_epochs"],
        seed=42 + f["fold_num"],
    )

    acc = accuracy_score(f["y_test_cls"], out_clf["clf_preds"])
    tabnet_clf["fold_acc"].append(acc)
    tabnet_clf["preds"].extend(out_clf["clf_preds"])
    tabnet_clf["probs"].append(out_clf["clf_probs"])
    tabnet_clf["true"].extend(f["y_test_cls"])
    tabnet_clf["quarters"].append(f["test_quarter"])

    mae  = mean_absolute_error(f["y_test_ret"], out_reg["reg_preds"])
    rmse = np.sqrt(mean_squared_error(f["y_test_ret"], out_reg["reg_preds"]))
    tabnet_reg["fold_mae"].append(mae)
    tabnet_reg["fold_rmse"].append(rmse)
    tabnet_reg["preds"].extend(out_reg["reg_preds"])
    tabnet_reg["true"].extend(f["y_test_ret"])

    train_time = time.time() - t0
    pred_dist = np.bincount(out_clf["clf_preds"].astype(int), minlength=3)

    print(f"Fold {f['fold_num']:2d} [{f['test_quarter']}]: "
          f"DA={acc:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  "
          f"pred_dist={pred_dist}  ({train_time:.0f}s)")


# ── Combined results across all 20 eval folds ──────────────────────
true  = np.array(tabnet_clf["true"])
preds = np.array(tabnet_clf["preds"])
probs = np.vstack(tabnet_clf["probs"])

print(f"\nTabNet (tuned) -- Avg DA:     {np.mean(tabnet_clf['fold_acc']):.4f}")
print(f"TabNet (tuned) -- Pooled F1:  {f1_score(true, preds, average='weighted'):.4f}")
print(f"TabNet (tuned) -- Pooled AUC: {roc_auc_score(true, probs, multi_class='ovr', average='weighted'):.4f}")

true_r  = np.array(tabnet_reg["true"])
preds_r = np.array(tabnet_reg["preds"])
print(f"\nTabNet (tuned) -- Avg MAE:  {np.mean(tabnet_reg['fold_mae']):.4f}")
print(f"TabNet (tuned) -- Avg RMSE: {np.mean(tabnet_reg['fold_rmse']):.4f}")
print(f"TabNet (tuned) -- Pooled R²: {r2_score(true_r, preds_r):.4f}")

print(f"\nPhase 1 config: {best_cfg_p1}")
print(f"Phase 2 config: {best_cfg_p2}")

print("\n" + classification_report(true, preds,
      target_names=["Low (<2%)", "Moderate (2-4%)", "Strong (≥4%)"]))
print("Confusion matrix:\n", confusion_matrix(true, preds))

# ── Tuning folds for completeness ───────────────────────────────────
print("\n--- Tuning folds (2020, used for HP selection *) ---")
for f in tune_folds_phase1:
    out = train_tabnet_clf(
        f,
        n_d=best_cfg_p1["n_d"], n_a=best_cfg_p1["n_a"],
        n_steps=best_cfg_p1["n_steps"], gamma=best_cfg_p1["gamma"],
        lambda_sparse=best_cfg_p1["lambda_sparse"],
        lr=best_cfg_p1["lr"], max_epochs=best_cfg_p1["max_epochs"],
        seed=42 + f["fold_num"],
    )
    acc = accuracy_score(f["y_test_cls"], out["clf_preds"])
    print(f"Fold {f['fold_num']:2d} [{f['test_quarter']}] *: DA={acc:.4f}")

Phase 1 tune:  ['2020_Q1', '2020_Q2', '2020_Q3', '2020_Q4']
Phase 1 eval:  ['2021_Q1', '2021_Q2', '2021_Q3', '2021_Q4', '2022_Q1', '2022_Q2', '2022_Q3', '2022_Q4']
Phase 2 tune:  folds 1-12 (2020_Q1 to 2022_Q4)
Phase 2 eval:  ['2023_Q1', '2023_Q2', '2023_Q3', '2023_Q4', '2024_Q1', '2024_Q2', '2024_Q3', '2024_Q4', '2025_Q1', '2025_Q2', '2025_Q3', '2025_Q4']

PHASE 1 — tuning on 4 folds (2020)


  0%|          | 0/24 [00:00<?, ?it/s]


Early stopping occurred at epoch 26 with best_epoch = 11 and best_val_logloss = 1.02814


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 30 with best_epoch = 15 and best_val_logloss = 1.03541


...

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 47 with best_epoch = 32 and best_val_logloss = 1.05388


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 60 with best_epoch = 45 and best_val_logloss = 1.05626


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


### Interim Save — Local Checkpoint

A local, secondary save of the tuned results (`models/tabnet_results.pkl`,
saved to the Colab runtime rather than Drive) — distinct from the
final save in Section 8. Includes a note on Fold 1: its validation
window (Dec 2019–Jan 2021) spans the COVID-19 shock and contains an
extreme +112% return absent from its training window, which inflates
the *internal* z-scored validation RMSE without affecting the
realized test-set MAE/RMSE reported elsewhere. Worth confirming
whether this cell is still needed, or a leftover from an earlier
debugging pass.

In [ ]:
os.makedirs("models", exist_ok=True)

true = np.array(tabnet_clf["true"]); preds = np.array(tabnet_clf["preds"])

pickle.dump({
    "clf": tabnet_clf,
    "reg": tabnet_reg,
    "feature_cols": flat_feature_cols,
    "classification_report": classification_report(
        true, preds, target_names=["Sell (<2%)", "Hold (2-4%)", "Buy (>=4%)"], output_dict=True
    ),
    "confusion_matrix": confusion_matrix(true, preds).tolist(),
    "note_fold1_regression": (
        "Fold 1's validation window (Dec 2019-Jan 2021) spans the COVID-19 market shock and "
        "contains an extreme +112% return event absent from its training window. This inflates "
        "the internal z-scored validation RMSE (scaled using train-period std) without affecting "
        "realized test-set MAE/RMSE, which are in line with other folds. Confirmed via SmoothL1Loss "
        "and multi-seed retest — see project notes."
    ),
}, open("models/tabnet_results.pkl", "wb"))

print("Saved models/tabnet_results.pkl")

Saved models/tabnet_results.pkl


## 6. Visualizations — Evaluation & Interpretation

Plots below use the pooled results of the tuned model across all 20
evaluation folds (`tabnet_clf`, `tabnet_reg`) from Section 5.

In [ ]:

# ── Quarterly DA over time ──────────────────────────────────────────
labels = tabnet_clf["quarters"]
x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(x, tabnet_clf["fold_acc"], marker="o", markersize=4, linewidth=1.5,
        color="#4C72B0", label="TabNet")
ax.axhline(np.mean(tabnet_clf["fold_acc"]), color="#4C72B0", linestyle=":",
           alpha=0.4, label=f"TabNet Avg ({np.mean(tabnet_clf['fold_acc']):.3f})")
ax.axhline(1/3, color="gray", linestyle="--", alpha=0.4, label="Random (0.333)")

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Directional Accuracy")
ax.set_title("Quarterly DA — TabNet Walk-Forward Evaluation (2021–2025)")
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Confusion Matrix

Row-normalized view of TabNet's per-class predictions — same layout
as the DNN notebook's confusion matrix, so the two are directly
comparable side by side.

In [ ]:
# ── Confusion Matrix ────────────────────────────────────────────────
cm = confusion_matrix(true, preds)
cm_pct = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=1)
labels = ["Low (<2%)", "Moderate (2-4%)", "Strong (≥4%)"]
ax.set_xticks(range(3)); ax.set_xticklabels(labels, rotation=20)
ax.set_yticks(range(3)); ax.set_yticklabels(labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — TabNet\n(row-normalized)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cm[i,j]}\n({cm_pct[i,j]:.0%})", ha="center", va="center",
                color="white" if cm_pct[i,j] > 0.5 else "black")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.show()

### Actual Return Distribution by Predicted Class

Boxplots of the true `target_return`, grouped by TabNet's predicted
class. A well-behaved classifier should show the "Predicted Strong"
box sitting clearly above the 4% threshold line, and boxes ordered
Low < Moderate < Strong.

In [ ]:
# ── Actual Return Distribution by Predicted Class ───────────────────
fig, ax = plt.subplots(figsize=(7, 5))
data_by_class = [true_r[preds == c] for c in [0, 1, 2]]
bp = ax.boxplot(data_by_class,
                tick_labels=["Predicted Low", "Predicted Moderate", "Predicted Strong"],
                showfliers=False, patch_artist=True)
for patch, color in zip(bp["boxes"], ["#C44E52", "#DDB967", "#55A868"]):
    patch.set_facecolor(color); patch.set_alpha(0.6)

for i, data in enumerate(data_by_class):
    median = np.median(data)
    ax.text(i + 1, median, f"{median:.3f}", ha="center", va="bottom",
            fontweight="bold", fontsize=9)

ax.axhline(0.02, color="gray", linestyle="--", alpha=0.5, label="2% threshold")
ax.axhline(0.04, color="gray", linestyle=":", alpha=0.5, label="4% threshold")
ax.set_ylabel("Actual peak return (target_return)")
ax.set_title("Actual Return Distribution by TabNet's Predicted Class")
ax.legend()
plt.tight_layout()
plt.show()

### Summary Table

Consolidates the headline numbers for TabNet — directional accuracy,
AUC, F1 for classification; R², MAE, RMSE for regression — plus both
tuning-phase configs, in the same format as the DNN notebook's summary
table for easy side-by-side comparison in the report.

In [ ]:
# ── Summary Table ───────────────────────────────────────────────────
print("=" * 60)
print("TABNET — FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"  Dataset:          ~24,000 earnings events (2014–2025)")
print(f"  Features:         {n_features} (technical + fundamental + sentiment + sector)")
print(f"  Architecture:     TabNet (attention-based, separate clf + reg)")
print(f"  Evaluation:       {len(tabnet_clf['fold_acc'])} quarterly walk-forward folds")
print(f"  HP optimization:  Optuna TPE, two-phase expanding window")
print(f"  Early stopping:   validation = quarter before test")
print(f"")
print(f"  CLASSIFICATION (3-class):")
print(f"    Avg DA:         {np.mean(tabnet_clf['fold_acc']):.4f}  (random baseline: 0.333)")
print(f"    Pooled AUC:     {roc_auc_score(true, probs, multi_class='ovr', average='weighted'):.4f}")
print(f"    Pooled F1:      {f1_score(true, preds, average='weighted'):.4f}")
print(f"    Strong (≥4%) F1: {f1_score(true, preds, average=None)[2]:.4f}")
print(f"")
print(f"  REGRESSION:")
print(f"    Pooled R²:      {r2_score(true_r, preds_r):.4f}")
print(f"    Avg MAE:        {np.mean(tabnet_reg['fold_mae']):.4f}")
print(f"    Avg RMSE:       {np.mean(tabnet_reg['fold_rmse']):.4f}")
print(f"")
print(f"  Phase 1 config:   {best_cfg_p1}")
print(f"  Phase 2 config:   {best_cfg_p2}")
print("=" * 60)

## 7. TabNet Attention Masks — Feature Importance

TabNet's biggest practical advantage over the DNN: attention weights
give per-sample feature importance for free, no SHAP or permutation
importance needed. Retrains once on the last evaluation fold (2023–2025
tuned config) purely to get a clean model to call `.explain()` on, then
reports the top 20 individual features by average attention, a bar
chart of the top 30, and a rollup of attention by modality
(Technical/VIX, Fundamental, Sentiment, Sector) — useful for arguing
which data sources TabNet actually leans on, independent of the DNN's
results.

In [ ]:

# Retrain on last fold to get a clean model reference
last_fold = eval_folds[-1] if 'eval_folds' in dir() else folds_data[-1]
out_last = train_tabnet_clf(
    last_fold,
    n_d=best_cfg_p2["n_d"], n_a=best_cfg_p2["n_a"],
    n_steps=best_cfg_p2["n_steps"], gamma=best_cfg_p2["gamma"],
    lambda_sparse=best_cfg_p2["lambda_sparse"],
    lr=best_cfg_p2["lr"], max_epochs=best_cfg_p2["max_epochs"],
    seed=42,
)

# Extract attention masks
explain_matrix, masks = out_last["model"].explain(last_fold["X_test"])

# ── Aggregate feature importance ────────────────────────────────────
avg_attention = explain_matrix.mean(axis=0)
sorted_idx = np.argsort(avg_attention)[::-1]

print("TOP 20 features by TabNet attention:")
print("-" * 55)
for i in sorted_idx[:20]:
    print(f"  {flat_feature_cols[i]:40s}  attn = {avg_attention[i]:.4f}")

# ── Plot top 30 ─────────────────────────────────────────────────────
top_n = 30
top_idx = sorted_idx[:top_n]
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(top_n), avg_attention[top_idx][::-1], color="#4C72B0", alpha=0.7)
ax.set_yticks(range(top_n))
ax.set_yticklabels([flat_feature_cols[i] for i in top_idx][::-1], fontsize=8)
ax.set_xlabel("Average Attention Weight")
ax.set_title(f"TabNet Feature Attention — Top {top_n} (fold {last_fold['test_quarter']})")
ax.grid(alpha=0.2, axis="x")
plt.tight_layout()
plt.show()

# ── Modality-level attention breakdown ──────────────────────────────
MODALITY_MAP = {}
for n in flat_feature_cols:
    if any(n.startswith(b + "_") or n == b for b in TECH_BASES + VIX_BASES):
        MODALITY_MAP[n] = "Technical/VIX"
    elif n.startswith("sector_"):
        MODALITY_MAP[n] = "Sector"
    elif n in ["pos_prob", "neg_prob", "overall_sentiment_score_pre",
               "ticker_sentiment_score_pre", "overall_sentiment_score_post",
               "ticker_sentiment_score_post"]:
        MODALITY_MAP[n] = "Sentiment"
    else:
        MODALITY_MAP[n] = "Fundamental"

modality_attention = {}
for mod in ["Technical/VIX", "Fundamental", "Sentiment", "Sector"]:
    cols_in_mod = [i for i, n in enumerate(flat_feature_cols) if MODALITY_MAP.get(n) == mod]
    modality_attention[mod] = {
        "total": avg_attention[cols_in_mod].sum(),
        "n_features": len(cols_in_mod),
        "avg_per_feature": avg_attention[cols_in_mod].mean() if cols_in_mod else 0,
    }

print("\nModality-level attention breakdown:")
print(f"{'Modality':>20s}  {'# Feats':>8s}  {'Total Attn':>10s}  {'Avg/Feature':>12s}")
print("-" * 55)
for mod, vals in sorted(modality_attention.items(), key=lambda x: -x[1]["total"]):
    print(f"{mod:>20s}  {vals['n_features']:>8d}  {vals['total']:>10.4f}  {vals['avg_per_feature']:>12.4f}")

## 8. Save Results

Pickles the tuned-phase configs and pooled classification/regression
outputs to Drive, keyed the same way as the DNN notebook's save cell
so both can be loaded together for the cross-pipeline comparison.

In [ ]:

save_dir = "/content/drive/MyDrive/models"
os.makedirs(save_dir, exist_ok=True)

results = {
    "best_cfg_p1": best_cfg_p1,
    "best_cfg_p2": best_cfg_p2,
    "tabnet_clf": tabnet_clf,
    "tabnet_reg": tabnet_reg,
    "eval_quarters": tabnet_clf["quarters"],
}

with open(f"{save_dir}/tabnet_results.pkl", "wb") as fh:
    pickle.dump(results, fh)
print(f"Results saved to {save_dir}/tabnet_results.pkl")